In [1]:
import pandas as pd

In [128]:
alpha1 = pd.read_csv("output_alpha/60min/alpha1.csv")
df_ret = pd.read_csv("return/60min/return.csv")
alpha1 = alpha1.set_index("datetime")
df_ret = df_ret.set_index("datetime")

In [129]:
def z_score(df):
    return (df - df.mean())/df.std()
def calculate_ic(alpha_norm,alpha_name):
    return df_ret[alpha_name].corr(alpha_norm)

In [223]:
import numpy as np
from scipy.stats import norm

def simpol_sigmoid(z, c, lam):
    """
    Sigmoid optimization function ψ(z) = cλ · (2Φ(λz) − 1)
    
    Args:
    z : array-like : input values
    c : float : scaling factor
    lam : float : lambda, controls the steepness
    
    Returns:
    array-like : transformed values using the sigmoid function
    """
    # Standard normal CDF Φ(λz)
    phi = norm.cdf(lam * z)
    
    # Compute ψ(z) = cλ · (2Φ(λz) − 1)
    psi = c * lam * (2 * phi - 1)
    
    return pd.Series(data=psi, index=z.index)

# Define the reverting sigmoid function
def reverting_sigmoid(x, c, lambd):
    """
    Implements the reverting sigmoid function for mean reversion.
    
    ψ(x) = -c * λ * x * exp(-λ² * x² / 2)
    
    Args:
    x : array-like : input data (e.g., returns)
    c : float : scaling factor
    lambd : float : lambda parameter (λ)
    
    Returns:
    array-like : transformed data using the reverting sigmoid function
    """
    return -c * lambd * x * np.exp(-lambd**2 * x**2 / 2)

import numpy as np


def sigmoid_signal(x, upper_threshold, lower_threshold):
    """
    Generate trading signals based on momentum thresholds.
    
    Args:
    momentum : array-like : momentum signal
    upper_threshold : float : threshold for entering long positions
    lower_threshold : float : threshold for entering short positions
    
    Returns:
    array-like : trading signal (+1 for long, -1 for short, 0 for no trade)
    """
    signals = x  #pd.Series(index=x.index)
    
    # Long signal: momentum above upper threshold
    signals[x > upper_threshold] = 1
    
    # Short signal: momentum below lower threshold
    signals[x < lower_threshold] = -1
    
    return signals


In [224]:
alpha_name_list = []
skew_initial_list = []
skew_reverse_list = []
skew_sigmoid_list = []
skew_simpol_list = []
ic_initial_list = []
ic_reverse_list = []
ic_sigmoid_list = []
ic_simpol_list = []

for alpha_name in alpha1.columns:
    alpha_norm = z_score(alpha1[alpha_name])
    alpha_name_list.append(alpha_name)
    skew_initial_list.append(alpha_norm.skew())
    
    alpha_reverse = reverting_sigmoid(alpha_norm,0.1,0.1)
    skew_reverse_list.append(alpha_reverse.skew())
    
    alpha_sigmoid = sigmoid_signal(alpha_norm,0.5,-0.5)
    skew_sigmoid_list.append(alpha_sigmoid.skew())
    
    alpha_simpol = simpol_sigmoid(alpha_norm,1,1)
    skew_simpol_list.append(alpha_simpol.skew())
    
    ic_initial_list.append(calculate_ic(alpha_norm,alpha_name))
    ic_reverse_list.append(calculate_ic(alpha_reverse,alpha_name))
    ic_sigmoid_list.append(calculate_ic(alpha_sigmoid,alpha_name))
    ic_simpol_list.append(calculate_ic(alpha_simpol,alpha_name))

In [225]:
alpha_simpol

datetime
2023-06-30 00:00:00    0.117042
2023-06-30 01:00:00   -0.051021
2023-06-30 02:00:00    0.234217
2023-06-30 03:00:00    0.176247
2023-06-30 04:00:00    0.002140
                         ...   
2023-12-28 20:00:00    0.235434
2023-12-28 21:00:00    0.240817
2023-12-28 22:00:00    0.682689
2023-12-28 23:00:00    0.682689
2023-12-29 00:00:00    0.682689
Length: 4369, dtype: float64

In [226]:
result_dict = {"alpha_name":alpha_name_list,
               "skew_initial":skew_initial_list,
               "ic_initial":ic_initial_list,
               "skew_reverse":skew_reverse_list,
               "ic_reverse":ic_reverse_list,
               "skew_sigmoid":skew_sigmoid_list,
               "ic_sigmoid":ic_sigmoid_list,
               "skew_simpol":skew_simpol_list,
               "ic_simpol":ic_simpol_list
               }

In [227]:
result_df = pd.DataFrame(result_dict)
result_df

,alpha_name,skew_initial,ic_initial,skew_reverse,ic_reverse,skew_sigmoid,ic_sigmoid,skew_simpol,ic_simpol
0,1ICHUSDT,1.195690,-0.043223,-0.347394,-0.050077,0.015874,-0.043223,0.016903,-0.043702
1,AAVEUSDT,0.800623,0.027911,-0.689459,-0.010928,0.092521,0.027911,0.096346,0.026394
2,ACCHUSDT,0.737089,-0.197299,-0.651616,0.218830,0.113656,-0.197299,0.119062,-0.196857
3,ADADABTC,1.151174,0.042356,-0.920345,-0.073697,0.120074,0.042356,0.123850,0.040695
4,ADADAETH,1.558249,-0.102968,-0.831666,0.079562,0.157713,-0.102968,0.162388,-0.104564
...,...,...,...,...,...,...,...,...,...
164,YFFIUSDT,-2.790571,-0.027912,1.169832,-0.059088,0.064309,-0.027912,0.068886,-0.030221
165,ZEECUSDT,0.144118,-0.253694,-0.129527,0.304552,0.035204,-0.253694,0.037805,-0.253115
166,ZEENUSDT,1.101316,-0.030938,-0.959328,0.023692,0.142072,-0.030938,0.149065,-0.032085
167,ZIILUSDT,1.912095,-0.204010,-0.355595,0.213130,0.085304,-0.204010,0.089390,-0.203881


In [228]:
len(result_df[abs(result_df['ic_initial'])>0.3])

16

In [229]:
len(result_df[abs(result_df['ic_reverse'])>0.3])

29

In [230]:
len(result_df[abs(result_df['ic_sigmoid'])>0.3])

16

In [231]:
len(result_df[abs(result_df['ic_simpol'])>0.3])

16

In [232]:
len(result_df[abs(result_df['ic_simpol'])>abs(result_df['ic_initial'])])

79

In [220]:
len(result_df[abs(result_df['ic_reverse'])>abs(result_df['ic_initial'])])

125

In [221]:
len(result_df[abs(result_df['ic_sigmoid'])>abs(result_df['ic_initial'])])

0